# 14 - Sensibilidad al umbral de viabilidad

Habíamos establecido un umbral del 50% sobre la viabilidad para establecer los conjuntos de dimensiones según el origen de captura de los datos.

Recordemos que la viabilidad se ha calculado de manera subjetiva en base a criterios del autor.

Vamos a realizar un análisis de sensibilidad con diferentes valores para el umbral: `config.THRESHOLDS` (40, 50, 60, 75 y 90).

Los resultados se generan ejecutando `make sensitivity` (`reports/results/sensitivity.csv`), con los hiperparámetros de `best_params.json` cuando se han calculado para el subconjunto.

Cuando un conjunto no tiene hiperparámetros propios (porque sus dimensiones no coinciden con ningún conjunto ajustado antes) se usan los de completo. La columna used_fallback marca esas filas.

El conjunto `completo` usa las 7 dimensiones, así que no depende del umbral y no lo recalculamos.

## Carga de datos

In [ ]:
from IPython.display import Markdown, display
import numpy as np
import pandas as pd

from urban_accessibility_ml.config import FEATURE_SETS, MATRIX_MODELS, PROFILES, REPORTS_DIR, THRESHOLDS
from urban_accessibility_ml.modeling.evaluate import summarize_folds

SENSITIVITY_FILE = REPORTS_DIR / "results" / "sensitivity.csv"
TUNED_FILE = REPORTS_DIR / "results" / "experiment-matrix-tuned.csv"
STRATEGIES = ["within_city", "mixed", "cross_city"]
CLEARLY_INFERIOR_RATIO = 0.8
SUBSETS = [feature_set for feature_set in FEATURE_SETS if feature_set != "completo"]

sensitivity = pd.read_csv(SENSITIVITY_FILE)
tuned = pd.read_csv(TUNED_FILE)
print(len(sensitivity), "filas de sensibilidad,", len(tuned), "filas de la matriz tuned")

## Conjuntos de variables por umbral
Vamos a mostrar a continuación cómo quedan los conjuntos de variables para cada umbral definido.

In [ ]:
summary = summarize_folds(sensitivity, ["threshold", "feature_set", "dimensions", "strategy", "profile", "model"])

sets = summary[["threshold", "feature_set", "dimensions"]].drop_duplicates()
sets.pivot(index="threshold", columns="feature_set", values="dimensions").reindex(columns=SUBSETS).fillna("(vacío)")

Observamos que el conjunto cartográfico aéreo está vacío a partir del 75% de umbral de viabilidad.

En el caso de 50% y 60%, el conjunto solo contiene cruces y mobiliario.
Y con un 40%: anchura, pavimento, cruces y mobiliario.

Mientras que el de calle tiene la anchura en todos los subconjuntos, incluso en el del 90%.
Después, el mobiliario se incluye con el 60% y el 75%.
Con el 50% se añaden cruces y señalización, mientras que con el 40% se incluye pendiente.

Observamos también que en el conjunto combinado, únicamente en los casos del 40% y del 60% se podría realizar el contraste de hipótesis H3.
En el caso del 40%, pavimento solo está en cartográfico aéreo, mientras que pendiente y cruces en calle.
En el caso del 60%, aéreo tiene cruces, y calle anchura.
En el resto de casos, calle tiene más dimensiones, por lo que el combinado es igual a calle (50%, 75% y 90%).

## Resumen por umbral, conjunto, perfil y modelo
Vamos a calcular para los conjuntos, la media y desviación típica entre pliegues por estrategia.

Comparamos cada fila se compara con la F1 macro del mismo modelo, estrategia y perfil con el conjunto completo, que es la misma para todos los umbrales.

In [ ]:
complete = summarize_folds(tuned[tuned["feature_set"] == "completo"], ["strategy", "profile", "model"])
complete = complete.rename(columns={"f1_macro_mean": "f1_macro_completo", "quadratic_weighted_kappa_mean": "qwk_completo"})
complete = complete[["strategy", "profile", "model", "f1_macro_completo", "qwk_completo"]]

n_rows = len(summary)
summary = summary.merge(complete, on=["strategy", "profile", "model"], validate="many_to_one")
assert len(summary) == n_rows, "algún modelo, estrategia o perfil no tiene referencia de completo"
summary["ratio"] = summary["f1_macro_mean"] / summary["f1_macro_completo"]
summary["supera_80"] = summary["ratio"] >= CLEARLY_INFERIOR_RATIO
summary.head()